In [3]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

verbose = 0
data_set = "gmtkn"

data_path_list = sorted(
    list(Path("../validate").glob(f"*3111142_{data_set}.csv")),
    key=lambda p: p.stat().st_ctime,
)
basis_args = "cc-pVDZ"
print(basis_args)

with open(f"../cc2cc/utils/{data_set}.json") as f:
    json_data = json.load(f)


full_subset_dict = {
    "sub1": [
        "W4_11",
        "G21EA",
        "G21IP",
        "DIPCS10",
        "PA26",
        "SIE4x4",
        "ALKBDE10",
        "YBDE18",
        "AL2X6",
        "HEAVYSB11",
        "NBPRC",
        "ALK8",
        "RC21",
        "G2RC",
        "BH76RC",
        "FH51",
        "TAUT15",
        "DC13",
    ],
    "sub2": [
        "MB16_43",
        "DARC",
        "RSE43",
        "BSR36",
        "CDIE20",
        "ISO34",
        "ISOL24",
        "C60ISO",
        "PArel",
    ],
    "sub3": [
        "BH76",
        "BHPERI",
        "BHDIV10",
        "INV24",
        "BHROT27",
        "PX13",
        "WCPT18",
    ],
    "sub4": [
        "RG18",
        "ADIM6",
        "S22",
        "S66",
        "HEAVY28",
        "WATER27",
        "CARBHB12",
        "PNICO23",
        "HAL59",
        "AHB21",
        "CHB6",
        "IL16",
    ],
    "sub5": [
        "IDISP",
        "ICONF",
        "ACONF",
        "Amino20x4",
        "PCONF21",
        "MCONF",
        "SCONF",
        "UPU23",
        "BUT14DIOL",
    ],
}
subset_list = []
for i_subset in full_subset_dict.keys():
    subset_list.extend(full_subset_dict[i_subset])

# accumulate summary dictionaries for each file
summary_list = []
subset_summary_list = []

for name_set, subset_list_ in full_subset_dict.items():
    summary_data_list = {}
    for i_subset in subset_list_:
        summary = {}

        for data_path in data_path_list:
            data = pd.read_csv(data_path)
            data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

            data_name = []
            data_reaction_energy_dft = []
            data_reaction_energy_ai = []

            reaction_dict = json_data[f"reaction-{i_subset}"]
            reaction_dict_copy = reaction_dict.copy()
            for i_reaction_name, i_reaction in reaction_dict_copy.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_ai = 0
                for i in range(len(systems_list)):
                    finished = True
                    mole_name = (
                        f"{i_subset}-{systems_list[i]}"
                        if i_subset != "BH76RC"
                        else systems_list[i]
                    )

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished = False
                        reaction_dict.pop(i_reaction_name)
                        break

                    col = data["name"] == mole_name
                    if col.any():
                        atomic_energy_dft += data[col]["error_dft_ene"].values[0] * int(
                            stoichiometry_list[i]
                        )
                        atomic_energy_ai += data[col]["error_scf_ene"].values[0] * int(
                            stoichiometry_list[i]
                        )
                        if verbose == 2:
                            print(
                                data[col]["error_dft_ene"].values[0],
                                int(stoichiometry_list[i]),
                                systems_list[i],
                            )
                    else:
                        finished = False
                        break

                if finished:
                    data_reaction_energy_dft.append(atomic_energy_dft)
                    data_reaction_energy_ai.append(atomic_energy_ai)
                    data_name.append(i_reaction_name)

            data_name = np.array(data_name)
            data_reaction_energy_dft = np.array(data_reaction_energy_dft)
            data_reaction_energy_ai = np.array(data_reaction_energy_ai)

            if verbose == 1:
                dft_error_argsort = np.argsort(data_reaction_energy_dft)[:5]
                ai_error_argsort = np.argsort(data_reaction_energy_ai)[:5]

                print("====DFT error====")
                print(
                    [
                        json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                        for i in dft_error_argsort
                    ]
                )
                print(data_reaction_energy_dft[dft_error_argsort])
                print("====AI error====")
                print(
                    [
                        json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                        for i in ai_error_argsort
                    ]
                )
                print(data_reaction_energy_ai[ai_error_argsort])

            summary.update(
                {
                    f"{data_path.stem} AI AE": (
                        np.mean(np.abs(data_reaction_energy_ai))
                        if len(data_reaction_energy_ai)
                        else 0
                    ),
                    f"{data_path.stem} DFT AE": (
                        np.mean(np.abs(data_reaction_energy_dft))
                        if len(data_reaction_energy_dft)
                        else 0
                    ),
                    f"{data_path.stem} Processed": f"{len(data_reaction_energy_dft)} / {len(reaction_dict)}",
                }
            )

            if f"{data_path.stem} AI AE" not in summary_data_list:
                summary_data_list[f"{data_path.stem} AI AE"] = data_reaction_energy_ai
            else:
                summary_data_list[f"{data_path.stem} AI AE"] = np.append(
                    summary_data_list[f"{data_path.stem} AI AE"],
                    data_reaction_energy_ai,
                )

            if f"{data_path.stem} DFT AE" not in summary_data_list:
                summary_data_list[f"{data_path.stem} DFT AE"] = data_reaction_energy_dft
            else:
                summary_data_list[f"{data_path.stem} DFT AE"] = np.append(
                    summary_data_list[f"{data_path.stem} DFT AE"],
                    data_reaction_energy_dft,
                )

            if f"{data_path.stem} reaction_dict" not in summary_data_list:
                summary_data_list[f"{data_path.stem} reaction_dict"] = len(
                    reaction_dict
                )
            else:
                summary_data_list[f"{data_path.stem} reaction_dict"] += len(
                    reaction_dict
                )

        summary_list.append(summary)

    subset_summary = {}
    for data_path in data_path_list:
        subset_summary.update(
            {
                f"{data_path.stem} AI AE": (
                    np.mean(np.abs(summary_data_list[f"{data_path.stem} AI AE"]))
                    if len(summary_data_list[f"{data_path.stem} AI AE"])
                    else 0
                ),
                f"{data_path.stem} DFT AE": (
                    np.mean(np.abs(summary_data_list[f"{data_path.stem} DFT AE"]))
                    if len(summary_data_list[f"{data_path.stem} DFT AE"])
                    else 0
                ),
                f"{data_path.stem} Processed": f"{len(summary_data_list[f"{data_path.stem} AI AE"])} / {summary_data_list[f"{data_path.stem} reaction_dict"]}",
            }
        )
    subset_summary_list.append(subset_summary)

# display one summary table for all files
header = pd.MultiIndex.from_product(
    [
        [data_path.stem for data_path in data_path_list],
        ["AI AE", "DFT AE", "Processed"],
    ],
    names=["data_path", "Type"],
)
df_summary = pd.DataFrame(
    subset_summary_list,
    index=full_subset_dict.keys(),
)
df_summary.columns = header
display(df_summary)

header = pd.MultiIndex.from_product(
    [
        [data_path.stem for data_path in data_path_list],
        ["AI AE", "DFT AE", "Processed"],
    ],
    names=["data_path", "Type"],
)
df_summary = pd.DataFrame(
    summary_list,
    index=subset_list,
)
df_summary.columns = header

df_summary.sort_values(
    by=(data_path.stem, "AI AE"),
    axis=0,
    inplace=True,
    ascending=False,
    # key=lambda x: float(x),
)
display(df_summary)

cc-pVDZ


data_path ccdft_cc-pVDZ_atom-1-3111142_gmtkn                      
Type                                   AI AE     DFT AE  Processed
sub1                                2.747119  23.441227  349 / 468
sub2                                1.756248   2.600035   55 / 243
sub3                                3.130680   7.442985  131 / 194
sub4                                1.948900   1.661153   98 / 246
sub5                                1.564405   0.639749    8 / 291

data_path ccdft_cc-pVDZ_atom-1-3111142_gmtkn                      
Type                                   AI AE     DFT AE  Processed
SIE4x4                             14.491921  21.908521    16 / 16
AL2X6                              13.398217   4.152005      3 / 6
HEAVYSB11                           8.124760   6.927331      5 / 7
PArel                               7.363893   1.813036     4 / 20
YBDE18                              7.299440   8.034205    12 / 18
DC13                                7.202449  11.735474     6 / 13
BHDIV10                             6.454916   5.566155     4 / 10
ALKBDE10                            5.181821  18.125250      9 / 9
FH51                                5.166205  11.106159     1 / 51
ALK8                                4.873804   3.337780      6 / 8
WCPT18                              4.576915   7.614363    17 / 18
BHPERI                              4.513475   4.555812     7 / 26
NBPRC                               4.332815   1.076943     5 / 12
PX13                                4.212978  10.171044     7 / 13
INV24                               3.743177   4.454271     7 / 24
WATER27                             3.702343   5.427502     7 / 27
BH76                                2.784290   9.182298    74 / 76
CDIE20                              2.752828   1.752673     2 / 20
PNICO23                             2.683026   0.820598    19 / 23
DIPCS10                             2.649730  12.310855    10 / 10
PA26                                2.437884   2.096682    17 / 26
S22                                 2.196647   1.016293     5 / 22
HAL59                               2.172058   1.879871    13 / 31
G2RC                                2.160532   5.854108    24 / 25
CHB6                                2.003839   2.700424      2 / 4
AHB21                               1.981230   2.558587    20 / 21
RC21                                1.944288   5.107387     1 / 21
ISO34                               1.769483   1.534374    14 / 34
ICONF                               1.564405   0.639749     8 / 17
G21EA                               1.460113   9.754524    25 / 25
G21IP                               1.327004   8.953336    36 / 36
W4_11                               1.277614  29.506869  140 / 140
S66                                 1.079267   0.753958     7 / 66
RSE43                               1.053133   3.164663    35 / 43
RG18                                1.013778   0.219384    16 / 18
BHROT27                             0.877795   0.637615    15 / 27
BH76RC                              0.849197  80.220383    30 / 30
CARBHB12                            0.830077   1.592103     9 / 12
TAUT15                              0.727573   2.925580     3 / 15
BSR36                               0.000000   0.000000     0 / 36
DARC                                0.000000   0.000000     0 / 14
MB16_43                             0.000000   0.000000     0 / 43
ISOL24                              0.000000   0.000000     0 / 24
HEAVY28                             0.000000   0.000000      0 / 0
ADIM6                               0.000000   0.000000      0 / 6
C60ISO                              0.000000   0.000000      0 / 9
IL16                                0.000000   0.000000     0 / 16
IDISP                               0.000000   0.000000      0 / 6
ACONF                               0.000000   0.000000     0 / 15
Amino20x4                           0.000000   0.000000     0 / 80
PCONF21                             0.000000   0.000000     0 / 18
MCONF                               0.000000   0.000000     0 / 51
SCONF                               0.000000   0.000000     0 / 17
UPU23                               0.000000   0.000000     0 / 23
BUT14DIOL                           0.000000   0.000000     0 / 64

In [16]:
df_summary

54

In [2]:
import numpy as np
(
    np.array([10.58423375510182, 225.43998315640636])
    - np.array([2.4392066220511355, 212.98927779812027])
)

# NBPRC-nh3-bh3

array([ 8.14502713, 12.45070536])

In [5]:
184 * 2.5

460.0

|File | AI AE | DFT AE | AI E | DFT E | AI Ele | DFT Ele | AI Dip | DFT Dip | Processed |
|---|---|---|---|---|---|---|---|---|---|
| atom-1-4049491 | 3.18 | 29.85 | 2.99 | 320.81 | 0.11 | 0.16 | 0.027 | 0.023 | 140 / 140 |
| atom-1-4049491 | 1.98 | 28.07 | 1.90 | 291.66 | 0.11 | 0.15 | 0.029 | 0.025 | 128 / 140 |
| atom-1-4049491 | 1.33 | 29.85 | 1.52 | 320.81 | 0.11 | 0.16 | 0.028 | 0.023 | 140 / 140 |
